In [0]:
dbutils.widgets.text("Catalog_name", "hms_catalog")
dbutils.widgets.text("Schema_name", "hms_silver")

Catalog_name = dbutils.widgets.get("Catalog_name")
Schema_name = dbutils.widgets.get("Schema_name")

In [0]:
check_catalog_availbility ={ rows[0] for rows in spark.sql(f"SHOW CATALOGS LIKE '{Catalog_name}'").collect()
}

if Catalog_name not in check_catalog_availbility:
    print(f"Catalog is not verified. Please create the catalog {Catalog_name}")
else:
    print(f"Catalog is verified. {Catalog_name}")

check_schema_availbility ={ rows[0] for rows in spark.sql(f"SHOW Schemas IN {Catalog_name}.{Schema_name}").collect()
}

if Catalog_name not in check_catalog_availbility:
    print(f"Schema is not verified. Please create the catalog {Schema_name}")
else:
    print(f"Schema is verified. {Schema_name}")



Catalog is verified. hms_catalog
Schema is verified. hms_silver


In [0]:
bookings = spark.table("hms_catalog.hms_bronze.bookings")
guests = spark.table("hms_catalog.hms_bronze.guests")
rooms = spark.table("hms_catalog.hms_bronze.rooms")
hotels = spark.table("hms_catalog.hms_bronze.hotels")
payments = spark.table("hms_catalog.hms_bronze.payments")
reviews = spark.table("hms_catalog.hms_bronze.reviews")


In [0]:
df_hms_booking_summary = bookings.join(guests, "guest_id", "inner")\
.join(rooms, "room_id", "inner")\
.join(hotels, "hotel_id", "inner")\
.join(payments, "booking_id", "inner")\
.join(reviews, "hotel_id", "inner")\
.select(

#booking Details
bookings['booking_id'], bookings['check_in'], bookings['check_out'], bookings['booking_status'],bookings['total_amount'], 

#guest Details
guests['guest_id'] , guests['guest_city'],guests['guest_age'],

#room Details
rooms['room_id'], rooms['room_type'], rooms['room_price'], rooms['room_status'], 

#hotel Details
hotels['hotel_id'], hotels['hotel_name'], hotels['hotel_city'], hotels['hotel_state'], hotels['hotel_rating'], hotels['hotel_phone'], 

#payment Details
payments['payment_id'], payments['payment_amount'], payments['payment_method'], payments['payment_status'], 

#review Details
reviews['review_id'], reviews['rating'], reviews['review_date'], reviews['review_comments'])
   


In [0]:
#create view for gold layer
df_hms_booking_summary.write.mode("overwrite").saveAsTable(f"{Catalog_name}.hms_gold.Vw_hms_booking_summary")